In [1]:
import hicstraw
import numpy as np
import math
import os

os.chdir('../..')
from load import generate_coverage_profile_intra_inter, load_GC_comtent, getmappability, load_num_res_site
from binlib import output_bin
from metric import calculate_correlation_with_groundtruth, calculate_precision_racall
from convert import convert_bin_to_bw

# back to current filder
os.chdir('Exps/exp9_trg_control_fit_seperate_model')

ref_genome_folder = '../../Data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY']
resolution = 50000


# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)



chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


### load bias

In [5]:
tgr_file = open('bias_allchroms_nonzeros_trg.txt','w+')
control_file = open('bias_allchroms_nonzeros_control.txt','w+')
tgr_file.write(f'chrom\tbin_num\tcutSitesNumber\tgcper\tmapscore\trowsum\n')
control_file.write(f'chrom\tbin_num\tcutSitesNumber\tgcper\tmapscore\trowsum\n')
with open('bias_with_control_allchroms_nonzeros.txt', 'r') as f:
    for line in f.readlines()[1:]:
        chrom, bin_num, control, cutSitesNumber, gcper, mapscore, trg = line.strip().split()
        if not float(cutSitesNumber) == float(mapscore) == float(gcper) == 0 :
            tgr_file.write(f'{chrom}\t{bin_num}\t{cutSitesNumber}\t{gcper}\t{mapscore}\t{trg}\n')
            control_file.write(f'{chrom}\t{bin_num}\t{cutSitesNumber}\t{gcper}\t{mapscore}\t{control}\n')

tgr_file.close()
control_file.close()

### Run script 

In [8]:
os.system('Rscript ../../GAM.R --input bias_allchroms_nonzeros_trg.txt --log trg_GAM_log --output trg_residual.txt')

Loading required package: nlme
This is mgcv 1.8-38. For overview type 'help("mgcv-package")'.


[1] "Loading input data ... "
[1] "Fitting GAM model ... "
[1] "Calculating residuals ..."
null device 
          1 
[1] "Done!"


0

In [9]:
os.system('Rscript ../../GAM.R --input bias_allchroms_nonzeros_control.txt --log control_GAM_log --output control_residual.txt')

Loading required package: nlme
This is mgcv 1.8-38. For overview type 'help("mgcv-package")'.


[1] "Loading input data ... "
[1] "Fitting GAM model ... "
[1] "Calculating residuals ..."
null device 
          1 
[1] "Done!"


0

### Merge residual to orig file and output to binfile

In [2]:

# init dict
trg_res_dict = dict()
control_res_dict = dict()
for key in chr_bin_number.keys():
    trg_res_dict[key] = np.zeros(chr_bin_number[key])
    control_res_dict[key] = np.zeros(chr_bin_number[key])

# merge trg
trg_bias_rowsum = open('bias_allchroms_nonzeros_trg.txt','r')
trg_residual = open('trg_residual.txt','r')
trg_bias_rowsum_lines = trg_bias_rowsum.readlines()[1:]
trg_residual_line = trg_residual.readlines()
for i, bias_line in enumerate(trg_bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = trg_residual_line[i]
    trg_res_dict[chr_][bin_num] = float(res)
trg_bias_rowsum.close()
trg_residual.close()

# merge control
control_bias_rowsum = open('bias_allchroms_nonzeros_control.txt','r')
control_residual = open('control_residual.txt','r')
control_bias_rowsum_lines = control_bias_rowsum.readlines()[1:]
control_residual_line = control_residual.readlines()
for i, bias_line in enumerate(control_bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = control_residual_line[i]
    control_res_dict[chr_][bin_num] = float(res)
control_bias_rowsum.close()
control_residual.close()

In [16]:
output_bin(trg_res_dict, trg_output_dir = 'exp9_trg_bin')
output_bin(control_res_dict, trg_output_dir = 'exp9_control_bin')

In [17]:
# output for visualization
convert_bin_to_bw('exp9_trg_bin', './exp9_trg.bw')
convert_bin_to_bw('exp9_control_bin', './exp9_control.bw')

### Test 1 : directly subtract residual

In [5]:
trg_sub_ctl_dict = dict()
for key in chr_bin_number.keys():
    trg_sub_ctl_dict[key] = np.subtract(trg_res_dict[key], control_res_dict[key])

In [14]:
output_bin(trg_sub_ctl_dict, trg_output_dir = 'exp9_test1_trg_sub_ctl_bin')
convert_bin_to_bw('exp9_test1_trg_sub_ctl_bin', './exp9_test1_trg_sub_ctl.bw')

(linux)
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test1_trg_sub_ctl_bin/_config.txt exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda3.seg --title HiC_trg_sub_ctl_lambda3 --fig exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda3.png --lambda 3 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test1_trg_sub_ctl_bin/_config.txt exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda2.seg --title HiC_trg_sub_ctl_lambda2 --fig exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda2.png --lambda 2 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test1_trg_sub_ctl_bin/_config.txt exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda1.seg --title HiC_trg_sub_ctl_lambda1 --fig exp9_test1_trg_sub_ctl_bin/exp9_test1_trg_sub_ctl_lambda1.png --lambda 1 --bootstrap

In [2]:
lambda_ = [1,2,3]

for l in lambda_:
    print(f'lambda {l}')
    calculate_correlation_with_groundtruth(f'test1/exp9_test1_trg_sub_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    calculate_precision_racall(f'test1/exp9_test1_trg_sub_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    print('\n')


lambda 1
Pearson corr. :  0.514
Spearman corr. :  0.75
Number of ground truth : 74
Number of target : 27
Number of hits : 25
precision : 0.926
recall : 0.338

hits by size : 
	 < 1Mb :  4
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  2
	10Mb - 20Mb :  2
	20Mb - 50Mb :  7
	>= 50Mb :  7
sum of  hit size : 758017909
average hit fraction : 0.875


lambda 2
Pearson corr. :  0.472
Spearman corr. :  0.689
Number of ground truth : 74
Number of target : 17
Number of hits : 16
precision : 0.941
recall : 0.216

hits by size : 
	 < 1Mb :  1
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  0
	10Mb - 20Mb :  2
	20Mb - 50Mb :  5
	>= 50Mb :  5
sum of  hit size : 527817918
average hit fraction : 0.824


lambda 3
Pearson corr. :  0.455
Spearman corr. :  0.678
Number of ground truth : 74
Number of target : 14
Number of hits : 12
precision : 0.857
recall : 0.162

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  0
	2Mb - 5Mb :  1
	5Mb - 10Mb :  0
	10Mb - 20Mb :  2
	20Mb - 50Mb :  4
	>= 50Mb :  5
sum of  hit size 

### Test 2 : pool then subtract

In [23]:
win_size = 5
pool_trg_res_dict = dict()
for chr_ in chr_bin_number.keys():
    pool_trg_res_dict[chr_] = np.zeros(chr_bin_number[chr_])
    for i in range(len(trg_res_dict[chr_])):
        start_ = i - math.floor(win_size/2) if i - math.floor(win_size/2) >=0 else 0 
        end_ = i + math.floor(win_size/2) if i + math.floor(win_size/2) <= chr_bin_number[chr_] else chr_bin_number[chr_]
        win_ = trg_res_dict[chr_][start_:end_+1]
        pool_trg_res_dict[chr_][i] = np.mean(win_)

pool_ctl_res_dict = dict()
for chr_ in chr_bin_number.keys():
    pool_ctl_res_dict[chr_] = np.zeros(chr_bin_number[chr_])
    for i in range(len(control_res_dict[chr_])):
        start_ = i - math.floor(win_size/2) if i - math.floor(win_size/2) >=0 else 0 
        end_ = i + math.floor(win_size/2) if i + math.floor(win_size/2) <= chr_bin_number[chr_] else chr_bin_number[chr_]
        win_ = control_res_dict[chr_][start_:end_+1]
        pool_ctl_res_dict[chr_][i] = np.mean(win_)

In [24]:
pool_trg_sub_ctl_dict = dict()
for key in chr_bin_number.keys():
    pool_trg_sub_ctl_dict[key] = np.subtract(pool_trg_res_dict[key], pool_ctl_res_dict[key])

In [25]:
output_bin(pool_trg_sub_ctl_dict, trg_output_dir = 'exp9_test2_pool_trg_sub_ctl_bin')
convert_bin_to_bw('exp9_test2_pool_trg_sub_ctl_bin', './exp9_test2_pool_trg_sub_ctl.bw')

(linux)
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test2_pool_trg_sub_ctl_bin/_config.txt exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda3.seg --title HiC_pooled_trg_sub_ctl_lambda3 --fig exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda3.png --lambda 3 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test2_pool_trg_sub_ctl_bin/_config.txt exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda2.seg --title HiC_pooled_trg_sub_ctl_lambda2 --fig exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda2.png --lambda 2 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test2_pool_trg_sub_ctl_bin/_config.txt exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda1.seg --title HiC_pooled_trg_sub_ctl_lambda1 --fig exp9_test2_pool_trg_sub_ctl_bin/exp9_test2_pooled_trg_sub_ctl_lambda1.png --lambda 1 --bootstrap

In [4]:
lambda_ = [1,2,3]

for l in lambda_:
    print(f'lambda {l}')
    calculate_correlation_with_groundtruth(f'test2/exp9_test2_pooled_trg_sub_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    calculate_precision_racall(f'test2/exp9_test2_pooled_trg_sub_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    print('\n')

lambda 1
Pearson corr. :  0.583
Spearman corr. :  0.742
Number of ground truth : 74
Number of target : 36
Number of hits : 29
precision : 0.806
recall : 0.392

hits by size : 
	 < 1Mb :  2
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  2
	10Mb - 20Mb :  2
	20Mb - 50Mb :  12
	>= 50Mb :  8
average hit size : 35200617.414
average hit fraction : 0.838


lambda 2
Pearson corr. :  0.587
Spearman corr. :  0.738
Number of ground truth : 74
Number of target : 28
Number of hits : 24
precision : 0.857
recall : 0.324

hits by size : 
	 < 1Mb :  1
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  1
	10Mb - 20Mb :  2
	20Mb - 50Mb :  9
	>= 50Mb :  8
average hit size : 37798662.917
average hit fraction : 0.836


lambda 3
Pearson corr. :  0.483
Spearman corr. :  0.684
Number of ground truth : 74
Number of target : 22
Number of hits : 16
precision : 0.727
recall : 0.216

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  0
	2Mb - 5Mb :  1
	5Mb - 10Mb :  0
	10Mb - 20Mb :  2
	20Mb - 50Mb :  7
	>= 50Mb :  6
average h

### Test 3 : fraction of control

In [3]:
# init dict
trg_res_dict = dict()
control_res_dict = dict()
for key in chr_bin_number.keys():
    trg_res_dict[key] = np.zeros(chr_bin_number[key])
    control_res_dict[key] = np.zeros(chr_bin_number[key])

# merge trg
trg_bias_rowsum = open('bias_allchroms_nonzeros_trg.txt','r')
trg_residual = open('trg_residual.txt','r')
trg_bias_rowsum_lines = trg_bias_rowsum.readlines()[1:]
trg_residual_line = trg_residual.readlines()
for i, bias_line in enumerate(trg_bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = trg_residual_line[i]
    trg_res_dict[chr_][bin_num] = float(res)
trg_bias_rowsum.close()
trg_residual.close()

# merge control
control_bias_rowsum = open('bias_allchroms_nonzeros_control.txt','r')
control_residual = open('control_residual.txt','r')
control_bias_rowsum_lines = control_bias_rowsum.readlines()[1:]
control_residual_line = control_residual.readlines()
for i, bias_line in enumerate(control_bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = control_residual_line[i]
    control_res_dict[chr_][bin_num] = float(res)
control_bias_rowsum.close()
control_residual.close()

In [12]:
fraction_list = [0.3, 0.5, 0.7]

for fraction in fraction_list:
    trg_sub_frac_ctl_dict = dict()
    for key in chr_bin_number.keys():
        frac_control_res_dict = control_res_dict[key] * fraction
        trg_sub_frac_ctl_dict[key] = np.subtract(trg_res_dict[key], frac_control_res_dict)

    output_bin(trg_sub_frac_ctl_dict, trg_output_dir = f'exp9_test3_trg_sub_{fraction}_ctl_bin')
    convert_bin_to_bw(f'exp9_test3_trg_sub_{fraction}_ctl_bin', f'./exp9_test3_trg_sub_{fraction}_ctl_bin.bw')

(linux)
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.3_ctl_bin/_config.txt exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda3.seg --title HiC_pooled_trg_sub_0.3_ctl_lambda3 --fig exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda3.png --lambda 3 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.3_ctl_bin/_config.txt exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda2.seg --title HiC_pooled_trg_sub_0.3_ctl_lambda2 --fig exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda2.png --lambda 2 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.3_ctl_bin/_config.txt exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda1.seg --title HiC_pooled_trg_sub_0.3_ctl_lambda1 --fig exp9_test3_trg_sub_0.3_ctl_bin/exp9_test3_trg_sub_0.3_ctl_lambda1.png --lambda 1 --bootstrap

../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.5_ctl_bin/_config.txt exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda3.seg --title HiC_pooled_trg_sub_0.5_ctl_lambda3 --fig exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda3.png --lambda 3 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.5_ctl_bin/_config.txt exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda2.seg --title HiC_pooled_trg_sub_0.5_ctl_lambda2 --fig exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda2.png --lambda 2 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.5_ctl_bin/_config.txt exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda1.seg --title HiC_pooled_trg_sub_0.5_ctl_lambda1 --fig exp9_test3_trg_sub_0.5_ctl_bin/exp9_test3_trg_sub_0.5_ctl_lambda1.png --lambda 1 --bootstrap

../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.7_ctl_bin/_config.txt exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda3.seg --title HiC_pooled_trg_sub_0.7_ctl_lambda3 --fig exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda3.png --lambda 3 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.7_ctl_bin/_config.txt exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda2.seg --title HiC_pooled_trg_sub_0.7_ctl_lambda2 --fig exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda2.png --lambda 2 --bootstrap
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp9_test3_trg_sub_0.7_ctl_bin/_config.txt exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda1.seg --title HiC_pooled_trg_sub_0.7_ctl_lambda1 --fig exp9_test3_trg_sub_0.7_ctl_bin/exp9_test3_trg_sub_0.7_ctl_lambda1.png --lambda 1 --bootstrap


In [18]:
for fraction in fraction_list :

    lambda_ = [1,2,3]
    print('>>>>>>> ', fraction)
    for l in lambda_:
        print(f'lambda {l}')
        calculate_correlation_with_groundtruth(f'test3/exp9_test3_trg_sub_{fraction}_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
        calculate_precision_racall(f'test3/exp9_test3_trg_sub_{fraction}_ctl_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
        print('\n')

>>>>>>>  0.3
lambda 1
Pearson corr. :  0.657
Spearman corr. :  0.862
Number of ground truth : 74
Number of target : 43
Number of hits : 31
precision : 0.721
recall : 0.419

hits by size : 
	 < 1Mb :  3
	1Mb - 2Mb :  0
	2Mb - 5Mb :  3
	5Mb - 10Mb :  3
	10Mb - 20Mb :  2
	20Mb - 50Mb :  13
	>= 50Mb :  7
sum of  hit size : 996367903
average hit fraction : 0.871


lambda 2
Pearson corr. :  0.66
Spearman corr. :  0.847
Number of ground truth : 74
Number of target : 37
Number of hits : 28
precision : 0.757
recall : 0.378

hits by size : 
	 < 1Mb :  2
	1Mb - 2Mb :  1
	2Mb - 5Mb :  3
	5Mb - 10Mb :  2
	10Mb - 20Mb :  1
	20Mb - 50Mb :  11
	>= 50Mb :  8
sum of  hit size : 963317906
average hit fraction : 0.859


lambda 3
Pearson corr. :  0.556
Spearman corr. :  0.846
Number of ground truth : 74
Number of target : 27
Number of hits : 20
precision : 0.741
recall : 0.27

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  0
	10Mb - 20Mb :  1
	20Mb - 50Mb :  9
	>= 50Mb :  7
sum